In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv("../content/Data.csv")

data = data.drop(columns=["VD", "VE", "tempo"])

data.index = (np.arange(0, len(data), 1).astype(float) * 0.07).round(5)

data = data.rename(columns={
    "Setpoint VD": "Wd",
    "Setpoint VE": "We",
    "Theta": "theta(Wd,We)",
    "X": "x(Wd,We)",
    "Y": "y(Wd,We)",
})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

PREDICTORS = ["Wd", "We"]
TARGET = "theta(Wd,We)"
colors = ['green', 'blue', 'orange', 'red', 'black',]


def PlotTimeSeries(df, img_name="data.svg", PlotOut=True):    
    # Estilo Seaborn
    sns.set_theme(style="whitegrid")

    # Criando a figura
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plotando Wd e We como linhas
    for preds in PREDICTORS:
        sns.lineplot(x=df.index, y=df[preds], ax=ax1, marker="o", label=preds,  linewidth=2)

    if PlotOut:
        ax2 = ax1.twinx()
        sns.lineplot(x=df.index, y=df[TARGET], ax=ax2, marker="o", label=TARGET, color="orange", linestyle="dashed", linewidth=2)
        ax2.set_ylabel("Posição Angular", fontsize=12, color="gray")

    # Ajustando os rótulos dos eixos
    ax1.set_xlabel("Tempo (s) - Período de amostragem (T=0.07 s)", fontsize=12)
    ax1.set_ylabel("Velocidade Angular", fontsize=12, color="black")

    # plt.xticks(df.index[::10], rotation=45, fontsize=1)  

    # Adicionando legendas
    ax1.legend(loc="upper left")
    if PlotOut:
        ax2.legend(loc="upper right")

    # Título
    plt.title("Posição angular em função da velocidade nas rodas", fontsize=14)

    # Salvando o gráfico
    plt.savefig(f"content/{img_name}", format="svg", dpi=600)

    # Exibindo o gráfico
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras import layers, models, regularizers, Sequential, initializers
from keras.callbacks import EarlyStopping
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error 

class ShuffleArchitecture:
    def __init__(self, input_size, hidden_sizes, output_size, act_h, act_o, param_reg):
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.output_size = output_size
        self.act_h = act_h
        self.act_o = act_o
        self.regularizer = regularizers.L2(param_reg)
        self.initializer = initializers.RandomUniform(minval=-0.5, maxval=0.5, seed=np.random.randint(1, 10000))
    
    def SetArchitecture(self):
        model = Sequential()
        self.model.add(layers.Input(shape=(self.input_size)))  # input layer

        for size in self.hidden_sizes[1:]:  # hidden layers
            self.model.add(layers.SimpleRNN(size,
                            activation=self.act_h,
                            kernel_regularizer=self.regularizer,
                            kernel_initializer=self.initializer,  
                            return_sequences=False,
                        ))

        self.model.add(layers.Dense(self.output_size,
                        activation=self.act_o,
                        kernel_regularizer=self.regularizer,
                        kernel_initializer=self.initializer,  
                        ))  # output layer

class PrepareData:
    def __init__(self, Dataset):
        self.df = Dataset.copy()
        self.scaler = StandardScaler()
        self.df[PREDICTORS] = self.scaler.fit_transform(self.df[PREDICTORS])
    
    def GetData(self, size=32):
        df = self.df.head(size)
        PlotTimeSeries(df = df)
        
        # Split into train (70%), validation (15%), and test (15%) sets
        train_data, temp_data = train_test_split(df, test_size=0.3, shuffle=False)
        valid_data, test_data = train_test_split(temp_data, test_size=0.5, shuffle=False)
            
        x_train = train_data[PREDICTORS].to_numpy()
        y_train = train_data[[TARGET]].to_numpy()

        x_val = valid_data[PREDICTORS].to_numpy()
        y_val = valid_data[[TARGET]].to_numpy()
        
        x_test = test_data[PREDICTORS].to_numpy()
        y_test = test_data[[TARGET]].to_numpy()
        
        x_sup = df[PREDICTORS]
        y_sup = df[TARGET]
        
        return x_train, y_train, x_val, y_val, x_test, y_test, x_sup, y_sup
    
class TrainWithSmallDataset:
    def __init__(self, Dataset):
        DataHandler = PrepareData(Dataset=Dataset)
        self.x_train, self.y_train, self.x_val, self.y_val, self.x_test, self.y_test, self.x_sup, self.y_sup = DataHandler.GetData()
        self.better_metrics = {}

            
    def CompileModel(self, model, epochs=100):
        model.compile(loss="mse", optimizer="adam")

        early_stop = EarlyStopping(monitor='val_loss',
                                   patience=10,
                                   restore_best_weights=True)

        self.history = model.fit(
            self.x_train, self.y_train,
            epochs=epochs,
            validation_data=(self.val_x, self.y_val),
            callbacks=[early_stop],
            verbose=False
        )
        
    def ComputeMetrics(self):
          # Calculando a saida com os dados normalizados
          train_pred = self.model.predict(self.x_train)
          test_pred = self.model.predict(self.x_test)
          val_pred = self.model.predict(self.x_val)
          sup_pred = self.model.predict(self.x_sup)

          metrics = {
                          'r2': r2_score(self.y_train, train_pred),
                          'r2_sup': r2_score(self.y_sup,  sup_pred),
                          'r2_test': r2_score(self.y_test, test_pred),
                          'r2_val': r2_score(self.y_val, val_pred),
                          'mse': mean_squared_error(self.y_train, train_pred),
                          'mse_sup':  mean_squared_error(self.y_sup,  sup_pred),
                          'mse_test': mean_squared_error(self.y_test, test_pred),
                          'mse_val': mean_squared_error(self.y_val, val_pred),
          }
          
          return(metrics)
      
    def Evaluate(self):
                
        self.GetData()    
        mse_values = []


        self.CompileModel(model, epochs=100)
        self.mse_results.append(np.array(mse_values))   